# PSPNet scratch run on Apple Silicon — paused for cloud handoff

This notebook records a new six-class Potsdam run initialized randomly, without a PSPNet checkpoint or ImageNet pretraining. It reached epoch 2 of 16 and is paused at `/Users/macbook/Data/runs/pspnet-mps-20261019/last.pt`.

The run uses the notebook's six validation scenes, 18 training scenes, and 14 held-out scenes, with 256×256 patches, batch size 8, and 10,000 sampled patches per epoch. Its target is 16 epochs.

Cell 3 is a single forward/backward smoke batch without an optimizer step. Cells 4 and 5 contain the local training and evaluation loops; leave them unexecuted. Running all cells would resume local training, which is paused by request. Continue from the epoch-2 checkpoint using the independent Colab and Kaggle runners. Checkpoints, logs, prepared data, and cloud results belong under `/Users/macbook/Data`, outside the repository.

In [1]:
import gc
import json
import os
import platform
import random
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torchvision
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "semantic_labeling").is_dir():
    raise RuntimeError(f"Expected the repository root as the working directory, got {PROJECT_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from semantic_labeling.constants import IGNORE_LABEL, INPUT_CHANNELS, N_CLASSES
from semantic_labeling.dataset import ISPRS_dataset
from semantic_labeling.inference import run_inference
from semantic_labeling.losses import CrossEntropy2d
from semantic_labeling.pspnet import PSPNet

DATA_ROOT = Path("/Users/macbook/Data/input")
POTSDAM_ROOT = DATA_ROOT / "potsdamvaihingen"
RUN_ID = "pspnet-scratch-mps-20261019-seed42"
RUN_DIR = Path("/Users/macbook/Data/runs/pspnet-mps-20261019")
MANIFEST_PATH = RUN_DIR / "run_manifest.json"

SEED = 42
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
WINDOW_SIZE = (256, 256)
BATCH_SIZE = 8
EPOCH_SIZE = 10_000
NUM_EPOCHS = 16
BASE_LR = 0.001
WEIGHT_DECAY = 0.0001
AUX_LOSS_WEIGHT = 0.4

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset root not found: {DATA_ROOT}")
if RUN_DIR.exists() and any(RUN_DIR.iterdir()) and not MANIFEST_PATH.is_file():
    raise FileExistsError(f"Refusing to reuse non-empty run directory without a manifest: {RUN_DIR}")
RUN_DIR.mkdir(parents=True, exist_ok=True)
if MANIFEST_PATH.is_file():
    previous_manifest = json.loads(MANIFEST_PATH.read_text())
    if previous_manifest.get("run_id") != RUN_ID:
        raise RuntimeError(f"Run directory belongs to another experiment: {previous_manifest.get('run_id')}")

print({
    "python": sys.executable,
    "python_version": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "device": str(DEVICE),
    "mps_available": torch.backends.mps.is_available(),
})

{'python': '/Users/macbook/Data/.venvs/deepnet-python314/bin/python', 'python_version': '3.14.8', 'torch': '2.14.1', 'torchvision': '0.29.1', 'device': 'mps', 'mps_available': True}


In [2]:
PARTICIPANT_LABEL_FILES = (
    POTSDAM_ROOT
    / "5_labels_for_participants/5_Labels_for_participants/top_potsdam_{}_label.tif"
)
DATA_FILES = POTSDAM_ROOT / "3_ortho_irrg/3_Ortho_IRRG/top_potsdam_{}_IRRG.tif"
DSM_FILES = POTSDAM_ROOT / "1_dsm/1_DSM/dsm_potsdam_0{}.tif"
NDSM_FILES = (
    POTSDAM_ROOT
    / "1_dsm_normalisation/1_DSM_normalisation/"
    "dsm_potsdam_0{}_normalized_lastools.jpg"
)
NO_BOUNDARY_LABEL_FILES = (
    DATA_ROOT
    / "label-all-potsdam-vaihingen/5_labels_all_noboundary/"
    "top_potsdam_{}_label_noBoundary.tif"
)

def scene_ids(directory, suffix):
    prefix = "top_potsdam_"
    return {
        path.name[len(prefix):-len(suffix)]
        for path in directory.glob(f"{prefix}*{suffix}")
    }

participant_ids = scene_ids(
    PARTICIPANT_LABEL_FILES.parent,
    "_label.tif",
)
all_ids = scene_ids(
    NO_BOUNDARY_LABEL_FILES.parent,
    "_label_noBoundary.tif",
)
VALID_IDS = ["3_12", "4_10", "4_11", "4_12", "5_11", "6_12"]
TRAIN_IDS = sorted(participant_ids - set(VALID_IDS))
TEST_IDS = sorted(all_ids - participant_ids)

assert len(participant_ids) == 24, f"Expected 24 participant-labeled tiles, got {len(participant_ids)}"
assert len(all_ids) == 38, f"Expected 38 full-reference tiles, got {len(all_ids)}"
assert set(VALID_IDS) <= participant_ids
assert len(TRAIN_IDS) == 18 and len(TEST_IDS) == 14
assert not (set(TRAIN_IDS) & set(VALID_IDS))
assert not (set(TRAIN_IDS) & set(TEST_IDS))
assert set(TRAIN_IDS) | set(VALID_IDS) | set(TEST_IDS) == all_ids

def dsm_scene_id(scene_id):
    first, second = scene_id.split("_")
    return f"{first}_{int(second):02d}"

DSM_IDS = [dsm_scene_id(scene_id) for scene_id in TRAIN_IDS]
for scene_id, dsm_id in zip(TRAIN_IDS, DSM_IDS):
    for path in (
        Path(str(DATA_FILES).format(scene_id)),
        Path(str(DSM_FILES).format(dsm_id)),
        Path(str(NDSM_FILES).format(dsm_id)),
        Path(str(PARTICIPANT_LABEL_FILES).format(scene_id)),
    ):
        if not path.is_file():
            raise FileNotFoundError(path)

print(f"Train ({len(TRAIN_IDS)}): {TRAIN_IDS}")
print(f"Validation ({len(VALID_IDS)}): {VALID_IDS}")
print(f"Held-out test ({len(TEST_IDS)}): {TEST_IDS}")

run_manifest = {
    "run_id": RUN_ID,
    "status": "prepared",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "git_revision": subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
    ).strip(),
    "python_executable": sys.executable,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "torchvision_version": torchvision.__version__,
    "device": str(DEVICE),
    "initialization": "random; no segmentation checkpoint or ImageNet weights",
    "seed": SEED,
    "train_ids": TRAIN_IDS,
    "validation_ids": VALID_IDS,
    "test_ids": TEST_IDS,
    "input_channels": INPUT_CHANNELS,
    "classes": N_CLASSES,
    "ignore_label": IGNORE_LABEL,
    "window_size": list(WINDOW_SIZE),
    "batch_size": BATCH_SIZE,
    "epoch_size": EPOCH_SIZE,
    "epochs": NUM_EPOCHS,
    "optimizer": "Adam",
    "learning_rate": BASE_LR,
    "weight_decay": WEIGHT_DECAY,
    "auxiliary_loss_weight": AUX_LOSS_WEIGHT,
    "input_scale": 255.0,
    "cache_tiles": True,
    "num_workers": 0,
    "dataset_root": str(DATA_ROOT),
    "run_directory": str(RUN_DIR),
}
if MANIFEST_PATH.is_file():
    assert json.loads(MANIFEST_PATH.read_text()).get("run_id") == RUN_ID
else:
    MANIFEST_PATH.write_text(json.dumps(run_manifest, indent=2) + "\n")

Train (18): ['2_10', '2_11', '2_12', '3_10', '3_11', '5_10', '5_12', '6_10', '6_11', '6_7', '6_8', '6_9', '7_10', '7_11', '7_12', '7_7', '7_8', '7_9']
Validation (6): ['3_12', '4_10', '4_11', '4_12', '5_11', '6_12']
Held-out test (14): ['2_13', '2_14', '3_13', '3_14', '4_13', '4_14', '4_15', '5_13', '5_14', '5_15', '6_13', '6_14', '6_15', '7_13']


In [3]:
train_dataset = ISPRS_dataset(
    TRAIN_IDS,
    DSM_IDS,
    data_files=str(DATA_FILES),
    label_files=str(PARTICIPANT_LABEL_FILES),
    dsm_files=str(DSM_FILES),
    ndsm_files=str(NDSM_FILES),
    cache=True,
    augmentation=True,
    window_size=WINDOW_SIZE,
    epoch_size=EPOCH_SIZE,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

# Exercise a real Potsdam batch through MPS forward and backward before the long run.
smoke_inputs, smoke_targets = next(iter(train_loader))
smoke_net = PSPNet(num_classes=N_CLASSES, use_aux=True).to(DEVICE)
smoke_net.train()
smoke_main, smoke_aux = smoke_net(smoke_inputs.to(DEVICE))
smoke_loss = CrossEntropy2d(smoke_main, smoke_targets.to(DEVICE)) + (
    AUX_LOSS_WEIGHT * CrossEntropy2d(smoke_aux, smoke_targets.to(DEVICE))
)
smoke_loss.backward()
print(
    "MPS smoke passed:",
    f"input={tuple(smoke_inputs.shape)}",
    f"main={tuple(smoke_main.shape)}",
    f"aux={tuple(smoke_aux.shape)}",
    f"loss={smoke_loss.item():.5f}",
)
del smoke_net, smoke_inputs, smoke_targets, smoke_main, smoke_aux, smoke_loss
gc.collect()

# Reset random streams so the smoke batch does not alter the recorded training seed.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

MPS smoke passed: input=(8, 5, 256, 256) main=(8, 6, 256, 256) aux=(8, 6, 256, 256) loss=10.85873


In [ ]:
def write_json_atomic(path, payload):
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(json.dumps(payload, indent=2) + "\n")
    os.replace(temporary_path, path)

def to_cpu_state(value):
    if torch.is_tensor(value):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {key: to_cpu_state(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_cpu_state(item) for item in value]
    if isinstance(value, tuple):
        return tuple(to_cpu_state(item) for item in value)
    return value


net = PSPNet(num_classes=N_CLASSES, use_aux=True).to(DEVICE)
optimizer = torch.optim.Adam(
    net.parameters(),
    lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
)
class_weights = torch.ones(N_CLASSES, dtype=torch.float32, device=DEVICE)
last_checkpoint_path = RUN_DIR / "last.pt"
completed_epoch = 0

if last_checkpoint_path.is_file():
    checkpoint = torch.load(
        last_checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )
    net.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    completed_epoch = checkpoint["epoch"]
    random.setstate(checkpoint["python_rng_state"])
    np.random.set_state(checkpoint["numpy_rng_state"])
    torch.set_rng_state(checkpoint["torch_rng_state"])
    if DEVICE.type == "mps" and checkpoint.get("mps_rng_state") is not None:
        torch.mps.set_rng_state(checkpoint["mps_rng_state"])
    print(f"Resuming from completed epoch {completed_epoch}")

run_manifest = json.loads(MANIFEST_PATH.read_text())
run_manifest["status"] = "running"
run_manifest["started_utc"] = run_manifest.get(
    "started_utc", datetime.now(timezone.utc).isoformat()
)
run_manifest["dsm_ids"] = DSM_IDS
run_manifest["batches_per_epoch"] = len(train_loader)
run_manifest["optimizer_steps_planned"] = NUM_EPOCHS * len(train_loader)
write_json_atomic(MANIFEST_PATH, run_manifest)

for epoch in range(completed_epoch + 1, NUM_EPOCHS + 1):
    epoch_started = time.perf_counter()
    net.train()
    rolling_loss = torch.zeros((), dtype=torch.float32, device=DEVICE)
    rolling_batches = 0
    epoch_loss_sum = 0.0

    for batch_index, (batch_inputs, batch_targets) in enumerate(train_loader, start=1):
        batch_inputs = batch_inputs.to(DEVICE)
        batch_targets = batch_targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        main_logits, auxiliary_logits = net(batch_inputs)
        loss = CrossEntropy2d(
            main_logits,
            batch_targets,
            weight=class_weights,
        ) + AUX_LOSS_WEIGHT * CrossEntropy2d(
            auxiliary_logits,
            batch_targets,
            weight=class_weights,
        )
        loss.backward()
        optimizer.step()

        rolling_loss.add_(loss.detach())
        rolling_batches += 1

        if batch_index % 100 == 0 or batch_index == len(train_loader):
            average_loss = rolling_loss.item() / rolling_batches
            epoch_loss_sum += average_loss * rolling_batches
            print(
                f"epoch {epoch:02d}/{NUM_EPOCHS} "
                f"batch {batch_index:04d}/{len(train_loader)} "
                f"window_loss={average_loss:.5f} "
                f"elapsed={time.perf_counter() - epoch_started:.0f}s",
                flush=True,
            )
            rolling_loss.zero_()
            rolling_batches = 0

    if DEVICE.type == "mps":
        torch.mps.synchronize()
    epoch_loss = epoch_loss_sum / len(train_loader)
    epoch_seconds = time.perf_counter() - epoch_started
    optimizer_state = to_cpu_state(optimizer.state_dict())
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": {
            name: value.detach().cpu()
            for name, value in net.state_dict().items()
        },
        "optimizer_state_dict": optimizer_state,
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "mps_rng_state": (
            torch.mps.get_rng_state().cpu()
            if DEVICE.type == "mps"
            else None
        ),
        "epoch_loss": epoch_loss,
        "run_id": RUN_ID,
    }
    temporary_checkpoint = RUN_DIR / "last.pt.tmp"
    torch.save(checkpoint, temporary_checkpoint)
    os.replace(temporary_checkpoint, last_checkpoint_path)

    if epoch % 4 == 0 or epoch == NUM_EPOCHS:
        model_path = RUN_DIR / f"model_epoch_{epoch:02d}.pt"
        torch.save(checkpoint["model_state_dict"], model_path)

    with (RUN_DIR / "training_log.jsonl").open("a", encoding="utf-8") as log_file:
        log_file.write(
            json.dumps(
                {
                    "epoch": epoch,
                    "mean_loss": epoch_loss,
                    "seconds": epoch_seconds,
                    "completed_utc": datetime.now(timezone.utc).isoformat(),
                }
            )
            + "\n"
        )

    run_manifest["last_completed_epoch"] = epoch
    run_manifest["last_epoch_loss"] = epoch_loss
    write_json_atomic(MANIFEST_PATH, run_manifest)
    print(
        f"completed epoch {epoch:02d}/{NUM_EPOCHS}: "
        f"mean_loss={epoch_loss:.5f}, elapsed={epoch_seconds / 60:.1f} min",
        flush=True,
    )

final_model_path = RUN_DIR / "final_model_state.pt"
torch.save(
    {name: value.detach().cpu() for name, value in net.state_dict().items()},
    final_model_path,
)
run_manifest["status"] = "trained"
run_manifest["training_completed_utc"] = datetime.now(timezone.utc).isoformat()
run_manifest["final_model"] = str(final_model_path)
write_json_atomic(MANIFEST_PATH, run_manifest)
print(f"Training complete. Final model: {final_model_path}")

In [ ]:
net.eval()

validation_accuracy = run_inference(
    net,
    VALID_IDS,
    data_files=str(DATA_FILES),
    dsm_files=str(DSM_FILES),
    ndsm_files=str(NDSM_FILES),
    eroded_label_files=str(NO_BOUNDARY_LABEL_FILES),
    stride=256,
    batch_size=6,
    window_size=WINDOW_SIZE,
)
test_accuracy = run_inference(
    net,
    TEST_IDS,
    data_files=str(DATA_FILES),
    dsm_files=str(DSM_FILES),
    ndsm_files=str(NDSM_FILES),
    eroded_label_files=str(NO_BOUNDARY_LABEL_FILES),
    stride=256,
    batch_size=6,
    window_size=WINDOW_SIZE,
)

metrics_path = RUN_DIR / "metrics.json"
metrics_payload = {
    "validation_accuracy_percent": validation_accuracy,
    "held_out_test_accuracy_percent": test_accuracy,
    "validation_ids": VALID_IDS,
    "test_ids": TEST_IDS,
    "metric": "overall pixel accuracy, ignoring label 6",
    "evaluated_utc": datetime.now(timezone.utc).isoformat(),
}
write_json_atomic(metrics_path, metrics_payload)

run_manifest = json.loads(MANIFEST_PATH.read_text())
run_manifest["status"] = "complete"
run_manifest["metrics_file"] = str(metrics_path)
run_manifest["metrics"] = metrics_payload
run_manifest["completed_utc"] = datetime.now(timezone.utc).isoformat()
write_json_atomic(MANIFEST_PATH, run_manifest)

print(json.dumps(metrics_payload, indent=2))